In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

In [2]:
# BartTorvik Dataset
barttorvik = pd.read_csv("Dataset/Barttorvik Away-Neutral.csv").sort_values(by = ['SEED', 'YEAR'], ascending = [True, False])
barttorvik_features = ['TEAM','YEAR', 'TEAM NO', 'SEED', 'BADJ O', 'BADJ D', 'EFG%','EFG%D', 'FTR', 'FTRD', 'FT%', 'TOV%', 'TOV%D', 'OREB%', 'DREB%', 
'2PT%', '2PT%D', '3PT%', '3PT%D', 'BLK%', 'BLKED%', 'AST%', 'OP AST%', '2PTR', '3PTR', '2PTRD', '3PTRD', 'BADJ T', 'EFF HGT', 'EXP', 'TALENT', 'PPPO', 'PPPD', 'ELITE SOS', 'WAB']
barttorvik = barttorvik[barttorvik_features]

# EvanMiya Dataset
evan_miya = pd.read_csv("Dataset/EvanMiya.csv").sort_values(by = ['SEED', 'YEAR'], ascending = [True, False])
evan_miya_features = ['TEAM', 'YEAR', 'TEAM NO', 'O RATE', 'D RATE','OPPONENT ADJUST', 'PACE ADJUST', 'TRUE TEMPO']
evan_miya = evan_miya[evan_miya_features]

# KenPom Dataset
kenpom = pd.read_csv("Dataset/KenPom Barttorvik.csv").sort_values(by = ['SEED', 'YEAR'], ascending = [True, False])
# Note: O, D, T very similar to barrtorvik - essentially same metric different derived using slightly different methods
kenpom_features = ['TEAM', 'YEAR', 'TEAM NO', 'KADJ O', 'KADJ D', 'KADJ T']  
kenpom = kenpom[kenpom_features]

# RPPF Dataset
rppf = pd.read_csv("Dataset/RPPF Ratings.csv").sort_values(by = ['SEED', 'YEAR'], ascending = [True, False])
rppf_features = ['TEAM', 'YEAR', 'TEAM NO', 'RADJ O', 'RADJ D', 'R PACE', 'R SOS', 'STROE', 'STRDE']
rppf = rppf[rppf_features]

# Shooting Splits Dataset
shooting_splits = pd.read_csv("Dataset/Shooting Splits.csv").sort_values(by = ['YEAR'], ascending = [False])
shooting_splits_features = ['TEAM', 'YEAR', 'TEAM NO', 'CLOSE TWOS FG%', 'CLOSE TWOS SHARE', 'CLOSE TWOS FG%D', 
'CLOSE TWOS D SHARE', 'FARTHER TWOS FG%', 'FARTHER TWOS SHARE', 'FARTHER TWOS FG%D', 'FARTHER TWOS D SHARE', 'THREES FG%',
'THREES SHARE', 'THREES FG%D', 'THREES D SHARE']
shooting_splits = shooting_splits[shooting_splits_features]

# Creating one df based on our merged team features
df = barttorvik.merge(evan_miya, on = ['TEAM NO', 'TEAM', 'YEAR'], how = 'inner')
df = df.merge(kenpom, on = ['TEAM NO', 'TEAM', 'YEAR'], how = 'inner')
df = df.merge(rppf, on = ['TEAM NO', 'TEAM', 'YEAR'], how = 'inner')
df = df.merge(shooting_splits, on = ['TEAM NO', 'TEAM', 'YEAR'], how = 'inner')
df = df.drop(columns = ['TEAM', 'YEAR'])
mm_team_features = df.groupby(['TEAM NO']).mean() # TEAM NO = key; represents TEAM + YEAR
mm_team_features

,SEED,BADJ O,BADJ D,EFG%,EFG%D,FTR,FTRD,FT%,TOV%,TOV%D,...,CLOSE TWOS FG%D,CLOSE TWOS D SHARE,FARTHER TWOS FG%,FARTHER TWOS SHARE,FARTHER TWOS FG%D,FARTHER TWOS D SHARE,THREES FG%,THREES SHARE,THREES FG%D,THREES D SHARE
TEAM NO,,,,,,,,,,,,,,,,,,,,,
332,5.0,106.3,87.9,45.8,45.2,34.2,30.5,62.7,16.5,16.4,...,53.7,33.3,37.0,29.9,34.2,41.8,33.0,40.1,28.9,24.9
333,9.0,105.9,91.8,48.8,46.4,32.9,46.8,70.8,20.3,19.2,...,53.8,27.8,34.6,31.3,35.6,35.7,33.9,35.3,32.2,36.6
334,16.0,98.8,100.9,48.1,48.9,34.8,41.3,66.3,23.5,20.6,...,60.9,30.6,36.2,30.4,36.8,38.0,32.7,36.2,32.4,31.3
335,9.0,106.2,91.5,47.4,47.4,47.6,41.8,73.7,23.7,21.4,...,51.8,30.5,35.2,29.8,34.2,33.5,33.2,35.3,37.2,36.0
336,5.0,114.8,96.8,52.0,51.7,31.8,43.7,68.2,18.6,25.0,...,61.1,40.4,40.3,30.7,33.6,26.6,35.1,35.7,32.9,33.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1143,4.0,120.7,96.3,50.1,49.4,33.9,32.5,80.4,17.0,15.6,...,55.8,29.5,38.4,25.6,38.5,28.6,32.4,34.6,33.6,41.9
1144,16.0,102.0,110.7,50.2,55.1,27.5,39.0,76.8,16.9,16.2,...,62.2,40.2,39.4,21.4,37.9,24.2,35.1,44.8,33.6,35.7
1145,16.0,100.5,107.7,45.6,50.5,28.4,37.4,68.8,14.2,17.9,...,57.7,39.0,31.5,20.5,37.4,21.6,32.9,42.9,33.4,39.5


In [3]:
# dataset of prior March Madness matchups
doubled_matchups = pd.read_csv("Dataset/Tournament Matchups.csv")

# updated to reflect the train/test dataset years
doubled_matchups = doubled_matchups[doubled_matchups.YEAR >= 2013] 

# only include the columns relevant to a binary (1/0) victory prediction
doubled_matchups_features = ['YEAR', 'TEAM NO', 'TEAM', 'SEED', 'SCORE']
doubled_matchups = doubled_matchups[doubled_matchups_features]

# Create a Game ID we can merge matchups on
doubled_matchups['GAME ID'] = doubled_matchups.index//2

# Series of transformations that change 2 adjacent rows/matchup --> 1 row/matchup
matchups = doubled_matchups.merge(doubled_matchups, how = 'inner', on = 'GAME ID')
matchups = matchups[matchups['TEAM_x'] != matchups['TEAM_y']]
matchups = matchups.reset_index()
matchups = matchups.drop(columns = ['index'])
matchups = matchups[matchups.index % 2 == 0]
matchups = matchups.reset_index()
matchups = matchups.drop(columns = ['index'])

# Data Cleaning: Having our data be in order --> Better Seeded Team + NO,  Worse Seeded Team + NO
matchups['Better_Seeded_Team'] = np.where(
    matchups['SEED_x'] <= matchups['SEED_y'], 
    matchups['TEAM_x'],
    matchups['TEAM_y']
)

matchups['BST_NO'] = np.where(
    matchups['SEED_x'] <= matchups['SEED_y'],
    matchups['TEAM NO_x'],
    matchups['TEAM NO_y']
)

matchups['Worse_Seeded_Team'] = np.where(
    matchups['SEED_x'] > matchups['SEED_y'],
    matchups['TEAM_x'],
    matchups['TEAM_y']
)

matchups['WST_NO'] = np.where(
    matchups['SEED_x'] > matchups['SEED_y'],
    matchups['TEAM NO_x'],
    matchups['TEAM NO_y']
)

# Label that will be used in the train-test process
matchups['BST_Won'] = ((matchups.SEED_x <= matchups.SEED_y) & (matchups.SCORE_x > matchups.SCORE_y)).astype(int) # if the team with the lower seed (better) has a higher score

# Data Cleaning: removing redundant/useless information:
matchups = matchups.drop(columns = ['TEAM NO_x', 'TEAM_x', 'SEED_x', 'SCORE_x', 'YEAR_y', 'TEAM NO_y', 'TEAM_y',	'SEED_y', 'SCORE_y']) 
matchups = matchups.rename(columns = {'YEAR_x': 'YEAR'}) # only one year
matchups

,YEAR,GAME ID,Better_Seeded_Team,BST_NO,Worse_Seeded_Team,WST_NO,BST_Won
0,2025,0,Auburn,1141,Alabama St.,1145,1
1,2025,1,Louisville,1119,Creighton,1134,0
2,2025,2,Michigan,1114,UC San Diego,1089,1
3,2025,3,Texas A&M,1092,Yale,1080,1
4,2025,4,Mississippi,1112,North Carolina,1104,1
...,...,...,...,...,...,...,...
750,2013,750,Florida,385,Michigan,366,0
751,2013,751,Marquette,369,Syracuse,341,0
752,2013,752,Louisville,370,Wichita St.,333,1
753,2013,753,Michigan,366,Syracuse,341,1


In [4]:
years = matchups['YEAR']
years

0      2025
1      2025
2      2025
3      2025
4      2025
       ... 
750    2013
751    2013
752    2013
753    2013
754    2013
Name: YEAR, Length: 755, dtype: int64

In [5]:
train_WL = matchups['BST_Won']
train_WL

0      1
1      0
2      1
3      1
4      1
      ..
750    0
751    0
752    1
753    1
754    1
Name: BST_Won, Length: 755, dtype: int64

In [6]:
mm_team_features.columns

Index(['SEED', 'BADJ O', 'BADJ D', 'EFG%', 'EFG%D', 'FTR', 'FTRD', 'FT%',
       'TOV%', 'TOV%D', 'OREB%', 'DREB%', '2PT%', '2PT%D', '3PT%', '3PT%D',
       'BLK%', 'BLKED%', 'AST%', 'OP AST%', '2PTR', '3PTR', '2PTRD', '3PTRD',
       'BADJ T', 'EFF HGT', 'EXP', 'TALENT', 'PPPO', 'PPPD', 'ELITE SOS',
       'WAB', 'O RATE', 'D RATE', 'OPPONENT ADJUST', 'PACE ADJUST',
       'TRUE TEMPO', 'KADJ O', 'KADJ D', 'KADJ T', 'RADJ O', 'RADJ D',
       'R PACE', 'R SOS', 'STROE', 'STRDE', 'CLOSE TWOS FG%',
       'CLOSE TWOS SHARE', 'CLOSE TWOS FG%D', 'CLOSE TWOS D SHARE',
       'FARTHER TWOS FG%', 'FARTHER TWOS SHARE', 'FARTHER TWOS FG%D',
       'FARTHER TWOS D SHARE', 'THREES FG%', 'THREES SHARE', 'THREES FG%D',
       'THREES D SHARE'],
      dtype='object')

In [7]:
differences = []

for _, team in matchups.iterrows():
    better_seeded_team_number = team['BST_NO']
    worse_seeded_team_number = team['WST_NO']

    # Calculate the difference in features
    better_seeded_team_stats = mm_team_features.loc[better_seeded_team_number]
    worse_seeded_team_stats = mm_team_features.loc[worse_seeded_team_number]

    # note: no need to flip signs --> model learns on its own
    diff_feature = better_seeded_team_stats - worse_seeded_team_stats # + means advantage for better seeded team; - means advantage for lower seeded team
    differences.append(diff_feature.to_list())

In [8]:
differences = np.array(differences)
differences

array([[-15. ,  32.6, -11.4, ...,  -2.3,  -4.2,  -4.7],
       [ -1. ,   8.5,  -2.7, ...,  -0.9,   1. ,   4.5],
       [ -7. ,   2.7,  -2.9, ...,  -7.1,  -1.8,  -5.4],
       ...,
       [ -8. ,   3.7, -10.8, ...,  -5.3,  -0.4,  -3.3],
       [  0. ,   4.2,   1.3, ...,   3.7,   4.3,  -6.8],
       [ -3. , -10.1, -15.2, ...,  -4.1,  -0.9,  -0.2]], shape=(755, 58))

In [9]:
diff_cols = [x + "_DIFF" for x in mm_team_features.columns.to_list()]
differences_df = pd.DataFrame(differences, columns = diff_cols)
differences_df.insert(loc = 0, column = "YEAR", value = years)
differences_df['BST_Won'] = train_WL
differences_df

,YEAR,SEED_DIFF,BADJ O_DIFF,BADJ D_DIFF,EFG%_DIFF,EFG%D_DIFF,FTR_DIFF,FTRD_DIFF,FT%_DIFF,TOV%_DIFF,...,CLOSE TWOS D SHARE_DIFF,FARTHER TWOS FG%_DIFF,FARTHER TWOS SHARE_DIFF,FARTHER TWOS FG%D_DIFF,FARTHER TWOS D SHARE_DIFF,THREES FG%_DIFF,THREES SHARE_DIFF,THREES FG%D_DIFF,THREES D SHARE_DIFF,BST_Won
0,2025,-15.0,32.6,-11.4,9.4,-1.9,2.7,5.6,4.8,-0.6,...,2.5,13.4,1.7,-3.6,2.2,3.9,-2.3,-4.2,-4.7,1
1,2025,-1.0,8.5,-2.7,3.4,0.3,0.0,9.9,0.7,-1.1,...,-0.6,-0.9,1.6,-1.4,-3.9,-0.9,-0.9,1.0,4.5,0
2,2025,-7.0,2.7,-2.9,-3.4,-1.8,-3.3,3.0,2.2,5.6,...,-5.8,4.7,1.3,2.5,11.3,-3.1,-7.1,-1.8,-5.4,1
3,2025,-9.0,-1.3,-13.2,-9.0,0.3,4.9,3.1,-3.7,3.8,...,-1.1,-13.7,-2.3,-1.7,2.2,-7.4,5.6,-0.3,-1.0,1
4,2025,-5.0,0.9,-3.7,-2.8,1.8,-0.1,11.8,1.6,-3.9,...,0.1,-0.5,8.4,-1.4,-6.0,-1.2,3.2,-0.8,6.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
750,2013,-1.0,-5.8,-7.6,1.8,-4.8,-3.6,4.3,-6.9,2.0,...,6.8,0.0,-8.9,-10.3,-6.2,-0.1,5.4,-1.7,-0.6,0
751,2013,-1.0,-5.2,-1.4,1.8,-1.1,-1.2,-12.6,3.2,3.9,...,3.0,5.4,-1.1,1.6,1.4,-3.8,-3.5,3.9,-4.4,0
752,2013,-8.0,3.7,-10.8,-1.2,-2.9,9.8,-8.9,1.6,-0.5,...,3.5,-1.6,-1.0,-2.7,-0.3,-0.5,-5.3,-0.4,-3.3,1
753,2013,0.0,4.2,1.3,5.8,3.1,-4.6,-17.9,6.1,-2.9,...,-5.7,6.3,1.0,10.5,12.5,4.5,3.7,4.3,-6.8,1


#### Splitting up our data into different years for the pre-2026 practice evaluation process; meant to find valid hp + avoid overfitting

In [10]:
# across all - Train-Test-Validation splits
train_differences_df = differences_df[(differences_df['YEAR'] >= 2013) & (differences_df['YEAR'] <= 2021)] 
validation_differences_df = differences_df[(differences_df['YEAR'] >= 2022) & (differences_df['YEAR'] <= 2023)] 
test_differences_df = differences_df[(differences_df['YEAR'] >= 2024) & (differences_df['YEAR'] <= 2025)] 

### Model 1: Logistic Regression + StandardScaling

In [11]:
"""Train"""
scaler = StandardScaler()

X_train = train_differences_df[diff_cols]
y_train = train_differences_df['BST_Won']

X_train_scaled = scaler.fit_transform(X_train)

# Variables used to find the best C value on the Validation set
c_range =  [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1]
validation_C_scores = {}
lowest_c_val = None

# Find the best hyperparameter for LogisticRegression (on validation data)
for c in c_range:
    model = LogisticRegression(C = c, max_iter = 1000)
    model.fit(X_train_scaled, y_train)

    """Validate"""
    X_val = validation_differences_df[diff_cols]
    y_val = validation_differences_df['BST_Won'] # ground truth
    
    X_val_scaled = scaler.transform(X_val) # Not training again: avoiding data-leakage
    
    validation_prob = model.predict_proba(X_val_scaled)[:, 1] # derive the probability that the better seeded team wins
    validation_brier = brier_score_loss(y_val, validation_prob)
    rounded_validation_brier = round(validation_brier, 4)
    
    print(f"C = {c}: VBS = {rounded_validation_brier}") # VBS = Validation Brier Score
    validation_C_scores[c] = rounded_validation_brier

    lowest_c_val = float(pd.Series(validation_C_scores).idxmin())

C = 0.05: VBS = 0.2486
C = 0.1: VBS = 0.2571
C = 0.15: VBS = 0.2617
C = 0.2: VBS = 0.2646
C = 0.25: VBS = 0.2667
C = 0.3: VBS = 0.2681
C = 0.35: VBS = 0.2692
C = 0.4: VBS = 0.2701
C = 0.45: VBS = 0.2708
C = 0.5: VBS = 0.2714
C = 0.55: VBS = 0.2719
C = 0.6: VBS = 0.2722
C = 0.65: VBS = 0.2726
C = 0.7: VBS = 0.273
C = 0.75: VBS = 0.2733
C = 0.8: VBS = 0.2735
C = 0.85: VBS = 0.2738
C = 0.9: VBS = 0.274
C = 0.95: VBS = 0.2741
C = 1: VBS = 0.2743


In [12]:
"""
Test - quality control check: need to ensure a valid Brier score here before actually testing
- want a model that does great on validation AND generalizes (reasonably well)
"""

# Refit with new train parameters
test_model = LogisticRegression(C = lowest_c_val, max_iter = 1000)
test_model.fit(X_train_scaled, y_train)

X_test = test_differences_df[diff_cols]
y_test = test_differences_df['BST_Won'] # ground truth

X_test_scaled = scaler.transform(X_test) # Not training again: avoiding data-leakage

test_prob = test_model.predict_proba(X_test_scaled)[:, 1] # derive the probability that the better seeded team wins
test_brier = brier_score_loss(y_test, test_prob)

print("Test Brier Score:", round(test_brier, 4))

test_prob_binary = np.round(test_prob)
test_correct = np.sum(test_prob_binary == y_test)

print(f"Games Predicted Correctly: {round(test_correct/len(test_prob_binary), 4)} on {len(test_prob_binary)} games")

Test Brier Score: 0.2021
Games Predicted Correctly: 0.6508 on 126 games


We see that in our LogisticRegression model, we have a Brier Score of 0.2021, with our chosen C of 0.05 from validation.

### Model 2: RandomForest Classifier

### Model 3: XGBoost Classifier

## Final Model Evaluation

 We will be training from 2013-2025 and testing on 2026 (this year's March Madness!) with the model that performed the best in pre-2026 evaluation.